In [1]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 57.4 MB/s eta 0:00:00


In [2]:
# anomaly_detection.py
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from rapidfuzz import fuzz
import re
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
DATABASE_URL = user_secrets.get_secret("DATABASE_URL_GLAMPING")

engine = create_engine(DATABASE_URL)

# ── Konfigurasi (ubah sesuai kebutuhan) ──────────────────────────────────────
SIMILARITY_THRESHOLD       = 75  # Teks (lexical): threshold kemiripan kata (0-100)
STYLE_SIMILARITY_THRESHOLD = 85  # Teks (stylometry): threshold kemiripan gaya (0-100)
                                  # PLACEHOLDER — WAJIB dikalibrasi pakai debug print.
RATING_SIMILARITY_THRESHOLD = 85 # Pola rating: % kategori rating yang identik (0-100)
                                  # CATATAN: skala ini beda dari HASH_SIMILARITY_THRESHOLD
                                  # versi lama (fuzz.ratio atas string) — sekarang murni
                                  # "persen posisi rating yang sama persis", bukan
                                  # kemiripan string. Angka 85 di sini perlu dikalibrasi
                                  # ulang, bukan otomatis setara dengan threshold lama.
MIN_KATA_TEKS               = 8  # Ulasan di bawah ini dilewati untuk similarity teks
WAKTU_MINIMAL_DETIK         = 8  # PLACEHOLDER — sesuaikan setelah lihat distribusi asli
WINDOW_JAM                  = 0.25  # rentang waktu berdekatan (jam)

# Sinyal jam tidak wajar HANYA relevan untuk Maribaya (day-trip, checkout sama hari).
AKTIFKAN_SINYAL_JAM     = False # Set True untuk Maribaya, False untuk Glamping
JAM_BUKA_WIB  = "08:00"
JAM_TUTUP_WIB = "17:15"

def _ke_menit(waktu_str):
    jam, menit = map(int, waktu_str.split(":"))
    return jam * 60 + menit

MENIT_BUKA  = _ke_menit(JAM_BUKA_WIB)
MENIT_TUTUP = _ke_menit(JAM_TUTUP_WIB)

# Filter tanggal — satu flag + satu helper (filter_tanggal_where) dipakai di
# load, reset, dan download, supaya ketiganya selalu konsisten.
FILTER_TANGGAL = False
START_DATE     = "2025-01-01"
END_DATE       = "2025-01-31"

def filter_tanggal_where():
    """Satu sumber kebenaran untuk klausa WHERE tanggal — dipakai di tiga
    tempat (load, reset, download) supaya tidak ada yang lupa disinkronkan.
    "1=1" dipakai sebagai basis netral saat filter nonaktif, supaya semua
    query bisa selalu menempel "AND {klausa}" tanpa perlu cek kondisi lagi."""
    if FILTER_TANGGAL:
        return ("submitted_at >= :start_date AND submitted_at < :end_date",
                 {"start_date": START_DATE, "end_date": END_DATE})
    return ("1=1", {})

RATING_COLS = [
    "rating_fasilitas", "rating_kebersihan",
    "rating_pelayanan_staff", "rating_skor_keseluruhan",
    "rating_harga", "rating_makanan",
    "rating_aktivitas", "rating_lokasi_tempat"
]

KOLOM_DIBUTUHKAN = ["id", "nama", "usia", "kota_asal", "submitted_at"] + RATING_COLS + ["ulasan_teks"]
KOLOM_SELECT      = ", ".join(KOLOM_DIBUTUHKAN)

# ── Load Data ─────────────────────────────────────────────────────────────────
print("Memuat data dari database...")

where_clause, where_params = filter_tanggal_where()
if FILTER_TANGGAL:
    print(f"  → Filter tanggal aktif: {START_DATE} s.d. {END_DATE}")

df = pd.read_sql(
    text(f"SELECT {KOLOM_SELECT} FROM reviews WHERE {where_clause}"),
    engine, params=where_params
)

df["submitted_at"] = pd.to_datetime(df["submitted_at"])
df["nama_norm"]    = df["nama"].str.strip().str.lower()
df["kota_norm"]    = df["kota_asal"].str.strip().str.lower()
df = df.sort_values("submitted_at").reset_index(drop=True)
ids = df["id"].to_numpy()

rating_tersedia = [c for c in RATING_COLS if c in df.columns]

def clean_text(t):
    if pd.isna(t):
        return ""
    t = t.lower()
    t = re.sub(r"[^\w\s]", "", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()

def extract_stylometry_features(text):
    """
    Fitur gaya tulisan dari TEKS MENTAH (tanda baca/kapital dipertahankan).
    Semua fitur murni counting, tidak ada pemahaman makna (beda embedding).
    """
    words   = text.split()
    n_words = len(words) if words else 1
    n_chars = len(text) if text else 1

    avg_word_len    = sum(len(w) for w in words) / n_words
    exclaim_ratio   = text.count("!") / n_chars
    uppercase_ratio = sum(1 for c in text if c.isupper()) / n_chars

    informal_markers = [
        "banget", "bgt", "deh", "sih", "dong", "yg", "ga", "gak",
        "nggak", "udah", "krn", "jd", "utk", "gitu", "aja", "kok"
    ]
    informal_count = sum(
        len(re.findall(r'\b' + re.escape(m) + r'\b', text.lower()))
        for m in informal_markers
    )
    informal_ratio = informal_count / n_words

    return {
        "avg_word_len":    avg_word_len,
        "exclaim_ratio":   exclaim_ratio,
        "uppercase_ratio": uppercase_ratio,
        "informal_ratio":  informal_ratio,
    }

def hitung_feature_ranges(list_fitur, persentil=5):
    ranges = {}

    for key in list_fitur[0]:
        nilai = np.array([f[key] for f in list_fitur], dtype=float)

        lo = np.percentile(nilai, persentil)
        hi = np.percentile(nilai, 100 - persentil)

        if hi <= lo:
            lo = nilai.min()
            hi = nilai.max()

        ranges[key] = (lo, hi)

    return ranges

def hitung_similarity_stylometry(vec_a, vec_b, feature_ranges):
    n = len(vec_a)
    jarak_kuadrat = 0
    for key in vec_a:
        lo, hi = feature_ranges[key]
        rentang = (hi - lo) if hi > lo else 1
        a_norm = min(max((vec_a[key] - lo) / rentang, 0), 1)  # clip 0-1 karena rentang dari persentil, bukan min-max absolut
        b_norm = min(max((vec_b[key] - lo) / rentang, 0), 1)
        jarak_kuadrat += (a_norm - b_norm) ** 2
    jarak = (jarak_kuadrat ** 0.5) / (n ** 0.5)
    return (1 - jarak) * 100

def batas_atas_lexical(len_a, len_b):
    """
    Batas atas MATEMATIS untuk fuzz.ratio/token_sort_ratio (rapidfuzz,
    berbasis Indel distance). indel_distance >= |len_a-len_b| SELALU —
    minimal butuh sejumlah itu insert/delete cuma buat menyamakan panjang.
    Jadi similarity TIDAK MUNGKIN melebihi 2*min(len_a,len_b)/(len_a+len_b).
    Ini bukan heuristik, ini bukti matematis dari formulanya sendiri — kalau
    batas ini sudah di bawah threshold, fuzz.token_sort_ratio() TIDAK PERLU
    dipanggil sama sekali, hasilnya sudah pasti gagal.
    """
    if len_a == 0 or len_b == 0:
        return 0
    return (2 * min(len_a, len_b) / (len_a + len_b)) * 100

def rating_similarity(tuple_a, tuple_b):
    """
    Perbandingan LANGSUNG posisi-ke-posisi — BUKAN fuzzy string edit-distance.
    Rating SELALU dalam urutan kolom yang sama di setiap baris, jadi tidak
    ada masalah alignment yang perlu diselesaikan fuzz.ratio(). Ini lebih
    murah (O(k) simple comparison, bukan algoritma DP string) DAN lebih
    tepat secara semantik — yang diukur memang "persen kategori rating
    yang identik", bukan "seberapa mirip sebagai untai karakter". Sebagai
    bonus, ini juga menghapus kekhawatiran sebelumnya soal baseline
    similarity kebetulan tinggi akibat alfabet kecil (digit 1-5 +
    underscore) di representasi string — representasi string itu sendiri
    sudah tidak dipakai lagi.

    Posisi yang salah satunya null di-skip dari perbandingan (bukan
    dianggap "cocok" atau "tidak cocok") — dua orang yang sama-sama
    skip rating_makanan (opsional) bukan berarti pola mereka identik.
    """
    pasangan_valid = [(a, b) for a, b in zip(tuple_a, tuple_b) if a is not None and b is not None]
    if not pasangan_valid:
        return 0
    sama = sum(1 for a, b in pasangan_valid if a == b)
    return (sama / len(pasangan_valid)) * 100

# ── Struktur Penyimpanan Hasil ────────────────────────────────────────────────
anomalies         = {}
signal_categories = {}

def flag(rid, reason, kategori):
    rid = int(rid)
    if rid not in anomalies:
        anomalies[rid] = []
    if reason not in anomalies[rid]:
        anomalies[rid].append(reason)
    if rid not in signal_categories:
        signal_categories[rid] = set()
    signal_categories[rid].add(kategori)

WINDOW_DETIK = WINDOW_JAM * 3600

# ── Sliding Window: Bangun pasangan dalam window waktu ───────────────────────
# Satu-satunya tempat pasangan dibangun — dipakai ULANG oleh SEMUA sinyal
# berbasis pasangan (identitas, teks, pola rating), tidak dibangun ulang
# per sinyal.
print(f"\nMembangun pasangan dalam window {WINDOW_JAM} jam...")

pasangan_window = []
for i in range(len(df)):
    for j in range(i + 1, len(df)):
        selisih = (df.loc[j, "submitted_at"] - df.loc[i, "submitted_at"]).total_seconds()
        if selisih > WINDOW_DETIK:
            break
        pasangan_window.append((i, j))

print(f"Total pasangan dalam window: {len(pasangan_window)}")

# Rating sebagai tuple (posisi tetap, BUKAN string) — dipakai Sinyal 4
df["rating_tuple"] = df[rating_tersedia].apply(
    lambda row: tuple(int(v) if pd.notna(v) else None for v in row),
    axis=1
)

nama_norm = df["nama_norm"].to_numpy()
kota_norm = df["kota_norm"].to_numpy()
usia = df["usia"].to_numpy()

nama_asli = df["nama"].to_numpy()
kota_asli = df["kota_asal"].to_numpy()

# ══════════════════════════════════════════════════════════════════════════
# SINYAL 1 — Identitas Berulang (nama + usia + kota, dalam window)
# ══════════════════════════════════════════════════════════════════════════
print(f"\n[SINYAL 1] Identitas berulang (nama+usia+kota) dalam {WINDOW_JAM} jam...")

for i, j in pasangan_window:

    if pd.isna(nama_norm[i]) or pd.isna(nama_norm[j]):
        continue
    if pd.isna(kota_norm[i]) or pd.isna(kota_norm[j]):
        continue
    if pd.isna(usia[i]) or pd.isna(usia[j]):
        continue

    if (
        nama_norm[i] == nama_norm[j]
        and kota_norm[i] == kota_norm[j]
        and usia[i] == usia[j]
    ):

        flag(
            ids[i],
            f"Identitas berulang: nama '{nama_asli[i]}' + usia {int(usia[i])} + kota '{kota_asli[i]}' sama dengan ID {ids[j]} dalam {WINDOW_JAM} jam",
            "identitas"
        )

        flag(
            ids[j],
            f"Identitas berulang: nama '{nama_asli[j]}' + usia {int(usia[j])} + kota '{kota_asli[j]}' sama dengan ID {ids[i]} dalam {WINDOW_JAM} jam",
            "identitas"
        )

# ══════════════════════════════════════════════════════════════════════════
# SINYAL 2 — Teks Ulasan Mirip (dalam window)
#
# Reuse pasangan_window langsung (poin a) — tidak membangun ulang pasangan
# via loop O(n²) terpisah, dan TIDAK perlu verifikasi window lagi karena
# pasangan_window sudah menjaminnya.
#
# Urutan evaluasi (poin b + c, sengaja dari yang paling murah):
#   1. Batas atas matematis panjang teks — skip fuzz.token_sort_ratio()
#      sama sekali kalau mustahil lolos threshold (bukan heuristik, bukti
#      matematis dari formula Indel distance).
#   2. Lexical (token_sort_ratio) — kalau SUDAH lolos, OR gate selesai,
#      stylometry tidak perlu dihitung sama sekali untuk pasangan ini.
#   3. Stylometry — cuma dipanggil kalau lexical gagal/di-skip. Ini yang
#      paling mahal (trigram + fitur), jadi ditaruh paling akhir dan
#      cuma jalan untuk MINORITAS pasangan yang lolos lexical dulu.
#
# feature_ranges dihitung dari SELURUH korpus (poin d), sekali di awal,
# bukan per-window — supaya similarity pasangan yang sama tidak berubah-
# ubah tergantung teks lain apa yang kebetulan ada di window itu.
# ══════════════════════════════════════════════════════════════════════════
print(f"[SINYAL 2] Teks ulasan mirip (kata: {SIMILARITY_THRESHOLD}, gaya: {STYLE_SIMILARITY_THRESHOLD}, min {MIN_KATA_TEKS} kata)...")

df["ulasan_clean"] = df["ulasan_teks"].fillna("").apply(clean_text)

mask_teks_valid = (
    df["ulasan_clean"].str.split().str.len() >= MIN_KATA_TEKS
)

teks_mentah_map = df.loc[mask_teks_valid, "ulasan_teks"].to_dict()
teks_clean_map  = df.loc[mask_teks_valid, "ulasan_clean"].to_dict()
fitur_map       = {idx: extract_stylometry_features(t) for idx, t in teks_mentah_map.items()}

if len(fitur_map) > 1:
    feature_ranges = hitung_feature_ranges(list(fitur_map.values()))
else:
    feature_ranges = None
    print("  → Tidak cukup ulasan (setelah gerbang panjang) untuk menghitung rentang fitur stylometry.")

for i, j in pasangan_window:
    if i not in teks_mentah_map or j not in teks_mentah_map:
        continue

    id_i = ids[i]
    id_j = ids[j]
    clean_i, clean_j = teks_clean_map[i], teks_clean_map[j]

    # 1. Prefilter matematis — skip fuzz.token_sort_ratio() kalau mustahil lolos
    lexical_sim   = None
    lolos_lexical = False
    if batas_atas_lexical(len(clean_i), len(clean_j)) >= SIMILARITY_THRESHOLD:
        lexical_sim   = fuzz.token_sort_ratio(clean_i, clean_j)
        lolos_lexical = lexical_sim >= SIMILARITY_THRESHOLD

        # Uncomment untuk debug
        # print(id_i, id_j, "lexical:", round(lexical_sim, 1), "|", clean_i, "<->", clean_j)

    if lolos_lexical:
        flag(id_i, f"Teks ulasan mirip dengan ID {id_j} (kemiripan kata {lexical_sim:.1f}%) dalam {WINDOW_JAM} jam", "teks")
        flag(id_j, f"Teks ulasan mirip dengan ID {id_i} (kemiripan kata {lexical_sim:.1f}%) dalam {WINDOW_JAM} jam", "teks")
        continue  # OR gate sudah terpenuhi, stylometry tidak perlu dihitung

    # 2. Lexical gagal/di-skip — coba stylometry (cuma sampai sini kalau perlu)
    if feature_ranges is None:
        continue

    style_sim = hitung_similarity_stylometry(fitur_map[i], fitur_map[j], feature_ranges)

    # Uncomment untuk debug — WAJIB dicek sebelum percaya STYLE_SIMILARITY_THRESHOLD
    # print(id_i, id_j, "style:", round(style_sim, 1), "|", clean_i, "<->", clean_j)

    if style_sim >= STYLE_SIMILARITY_THRESHOLD:
        flag(id_i, f"Teks ulasan mirip dengan ID {id_j} (kemiripan gaya tulisan {style_sim:.1f}%) dalam {WINDOW_JAM} jam", "teks")
        flag(id_j, f"Teks ulasan mirip dengan ID {id_i} (kemiripan gaya tulisan {style_sim:.1f}%) dalam {WINDOW_JAM} jam", "teks")

# ══════════════════════════════════════════════════════════════════════════
# SINYAL 3 — Jam Tidak Wajar (khusus Maribaya, nonaktif di file ini)
# ══════════════════════════════════════════════════════════════════════════
if AKTIFKAN_SINYAL_JAM:
    print(f"[SINYAL 3] Submission di luar jam operasional ({JAM_BUKA_WIB}–{JAM_TUTUP_WIB} WIB)...")

    submitted_wib = df["submitted_at"] + pd.Timedelta(hours=7)
    menit_wib     = submitted_wib.dt.hour * 60 + submitted_wib.dt.minute

    # Di luar [MENIT_BUKA, MENIT_TUTUP] — sebelum 08:00 ATAU sesudah 17:15.
    # Dihitung dalam menit (bukan cuma jam) supaya 17:15 presisi, bukan
    # ke-bulatkan jadi "jam 17" yang gagal bedakan 17:00 dari 17:45.
    mask_tidak_wajar = (menit_wib < MENIT_BUKA) | (menit_wib > MENIT_TUTUP)

    for idx in df[mask_tidak_wajar].index:
        jam_tampil = submitted_wib.loc[idx]
        flag(df.loc[idx, "id"],
             f"Submission di jam tidak wajar ({jam_tampil.strftime('%Y-%m-%d %H:%M')} WIB) — di luar jam operasional {JAM_BUKA_WIB}–{JAM_TUTUP_WIB}",
             "jam")
else:
    print("[SINYAL 3] Jam tidak wajar — dinonaktifkan (mode glamping, checkout bisa kapan saja).")

# ══════════════════════════════════════════════════════════════════════════
# SINYAL 4 — Pola Rating Mirip (perbandingan posisi langsung, bukan fuzzy string)
# ══════════════════════════════════════════════════════════════════════════
print(f"[SINYAL 4] Pola rating mirip (threshold: {RATING_SIMILARITY_THRESHOLD})...")

rating_tuples = df["rating_tuple"].to_numpy()

for i, j in pasangan_window:
    similarity = rating_similarity(
        rating_tuples[i],
        rating_tuples[j]
    )
    
    id_i = ids[i]
    id_j = ids[j]

    # Uncomment untuk debug
    # print(id_i, id_j, similarity, rating_tuples[i], "<->", rating_tuples[j])

    if similarity < RATING_SIMILARITY_THRESHOLD:
        continue

    flag(id_i, f"Pola rating sangat mirip dengan ID {id_j} (kemiripan: {similarity:.1f}%) dalam {WINDOW_JAM} jam", "pola_rating")
    flag(id_j, f"Pola rating sangat mirip dengan ID {id_i} (kemiripan: {similarity:.1f}%) dalam {WINDOW_JAM} jam", "pola_rating")

# ══════════════════════════════════════════════════════════════════════════
# SINYAL 5 — Submit Terlalu Cepat Sejak Token Dibuat
# KETERBATASAN: /scan menghapus token > 1 hari — review lama yang tokennya
# sudah dibersihkan tidak bisa dicek sinyal ini (LEFT JOIN null, dilewati).
# ══════════════════════════════════════════════════════════════════════════
print(f"\n[SINYAL 5] Submit kurang dari {WAKTU_MINIMAL_DETIK} detik sejak token dibuat...")

where_clause_waktu, where_params_waktu = filter_tanggal_where()
df_waktu = pd.read_sql(
    text(f"""
        SELECT r.id, r.submitted_at, t.created_at AS token_created_at
        FROM reviews r
        LEFT JOIN tokens t ON r.token = t.token
        WHERE {where_clause_waktu.replace('submitted_at', 'r.submitted_at')}
    """),
    engine, params=where_params_waktu
)

df_waktu["submitted_at"]     = pd.to_datetime(df_waktu["submitted_at"])
df_waktu["token_created_at"] = pd.to_datetime(df_waktu["token_created_at"])

token_hilang = df_waktu["token_created_at"].isna().sum()
if token_hilang > 0:
    print(f"  → PERHATIAN: {token_hilang} review tidak punya token yang cocok (kemungkinan "
          f"sudah dibersihkan cleanup harian di /scan) — sinyal ini tidak bisa dicek untuk review tsb.")

df_waktu["selisih_detik"] = (df_waktu["submitted_at"] - df_waktu["token_created_at"]).dt.total_seconds()

terlalu_cepat = df_waktu[
    df_waktu["token_created_at"].notna() &
    (df_waktu["selisih_detik"] < WAKTU_MINIMAL_DETIK)
]

for _, row in terlalu_cepat.iterrows():
    flag(int(row["id"]),
         f"Submit hanya {row['selisih_detik']:.1f} detik setelah token dibuat — mustahil untuk pengisian manual",
         "cepat")

# ══════════════════════════════════════════════════════════════════════════
# TIER KEYAKINAN
#   KUAT     : jam (mutlak) | cepat (mutlak) | identitas+teks | identitas+pola_rating
#   MENENGAH : identitas saja | pola_rating+teks (identitas beda)
#   RENDAH   : pola_rating saja | teks saja
# ══════════════════════════════════════════════════════════════════════════
def hitung_tier(kategoris):
    if "jam" in kategoris:
        return "KUAT"
    if "cepat" in kategoris:
        return "KUAT"
    if "identitas" in kategoris and "teks" in kategoris:
        return "KUAT"
    if "identitas" in kategoris and "pola_rating" in kategoris:
        return "KUAT"

    if kategoris == {"identitas"}:
        return "MENENGAH"
    if kategoris == {"pola_rating", "teks"}:
        return "MENENGAH"

    if kategoris == {"pola_rating"}:
        return "RENDAH"
    if kategoris == {"teks"}:
        return "RENDAH"

    return "RENDAH"  # safety net — seharusnya tidak pernah kena given kombinasi di atas

tier_per_id = {rid: hitung_tier(kats) for rid, kats in signal_categories.items()}

# ── Update Database ───────────────────────────────────────────────────────────
# anomaly_category (KUAT/MENENGAH/RENDAH) dan anomaly_reason (daftar alasan)
# dipisah jadi dua kolom — sebelumnya digabung dalam satu string "TIER: X :: ...",
# menyulitkan GROUP BY/filter per tier tanpa parsing string dulu.
#
# SEBELUM DIJALANKAN, kolom ini perlu ditambahkan ke database (sekali saja):
#   ALTER TABLE reviews ADD COLUMN anomaly_category VARCHAR;
print(f"\nTotal submission ter-flag: {len(anomalies)}")

where_clause_reset, where_params_reset = filter_tanggal_where()

with engine.connect() as conn:
    conn.execute(text(f"""
        UPDATE reviews
        SET is_anomaly = FALSE, anomaly_category = NULL, anomaly_reason = NULL
        WHERE {where_clause_reset}
    """), where_params_reset)

    for rid, reasons in anomalies.items():
        tier        = tier_per_id[rid]
        reason_text = " | ".join(reasons)
        conn.execute(text("""
            UPDATE reviews
            SET is_anomaly = TRUE, anomaly_category = :category, anomaly_reason = :reason
            WHERE id = :id
        """), {"id": rid, "category": tier, "reason": reason_text})

    conn.commit()

print("Database berhasil diperbarui.")

# ── Tampilkan & Download Tabel Anomali ────────────────────────────────────────
print("\n── Tabel Anomali ──────────────────────────────────────────────")

kolom_tampil = ["id", "nama", "kota_asal", "submitted_at"] + rating_tersedia + \
               ["ulasan_teks", "is_anomaly", "anomaly_category", "anomaly_reason"]
kolom_query  = ", ".join(kolom_tampil)

where_clause_dl, where_params_dl = filter_tanggal_where()
df_hasil = pd.read_sql(
    text(f"""
        SELECT {kolom_query} FROM reviews
        WHERE is_anomaly = TRUE AND {where_clause_dl}
        ORDER BY id
    """),
    engine, params=where_params_dl
)

if df_hasil.empty:
    print("Tidak ada anomali ditemukan.")
else:
    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_colwidth", 80)
    pd.set_option("display.width", 200)
    print("=" * 70)
    print("HASIL DETEKSI ANOMALI")
    print("=" * 70)
    print(f"Total submission anomali : {len(df_hasil)}")

    for _, row in df_hasil.iterrows():
        print("\n" + "=" * 70)
        print(f"ANOMALI ID {row['id']}")
        print("=" * 70)

        print(f"Nama         : {row['nama']}")
        print(f"Kota         : {row['kota_asal']}")
        print(f"Waktu Submit : {row['submitted_at']}")

        print("\nRating")
        print("-" * 20)
        for col in rating_tersedia:
            nilai = row[col]
            teks_bintang = "-" if pd.isna(nilai) else "⭐" * int(nilai)
            print(f"{col.replace('rating_','').replace('_',' ').title():20}: {teks_bintang}")

        print("\nUlasan")
        print("-" * 20)
        print("-" if pd.isna(row["ulasan_teks"]) else row["ulasan_teks"])

        print(f"\nTingkat Keyakinan: {row['anomaly_category']}")
        print("Alasan Anomali")
        print("-" * 20)
        alasan = [a.strip() for a in row["anomaly_reason"].split(" | ")]
        for i, a in enumerate(alasan, 1):
            print(f"{i}. {a}")

    output_path = "anomali_reviews.csv"
    df_hasil.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"\nTabel anomali disimpan ke: {output_path}")
    print(f"Total anomali: {len(df_hasil)} submission")

Memuat data dari database...

Membangun pasangan dalam window 0.25 jam...
Total pasangan dalam window: 10

[SINYAL 1] Identitas berulang (nama+usia+kota) dalam 0.25 jam...
[SINYAL 2] Teks ulasan mirip (kata: 75, gaya: 85, min 8 kata)...
[SINYAL 3] Jam tidak wajar — dinonaktifkan (mode glamping, checkout bisa kapan saja).
[SINYAL 4] Pola rating mirip (threshold: 85)...

[SINYAL 5] Submit kurang dari 8 detik sejak token dibuat...
  → PERHATIAN: 8 review tidak punya token yang cocok (kemungkinan sudah dibersihkan cleanup harian di /scan) — sinyal ini tidak bisa dicek untuk review tsb.

Total submission ter-flag: 4
Database berhasil diperbarui.

── Tabel Anomali ──────────────────────────────────────────────
HASIL DETEKSI ANOMALI
Total submission anomali : 4

ANOMALI ID 1
Nama         : Bernardo
Kota         : A
Waktu Submit : 2026-07-03 06:44:26.683123

Rating
--------------------
Fasilitas           : ⭐⭐⭐⭐⭐
Kebersihan          : ⭐⭐⭐⭐⭐
Pelayanan Staff     : ⭐⭐⭐⭐⭐
Skor Keseluruhan    : ⭐⭐⭐